In [ ]:
_NB_NAME_ = "Bias-Variance Tradeoff"
# @title 🤖 AI Feedback — fill in your username and click ▶ Run
# @markdown No setup needed — AI grading runs directly in Colab for free.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ─────────────────────────────────────────────────────────────
import json, textwrap, re as _re
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("\u26a0  Run the quiz cell first, then re-run this cell.")
else:
    nd = sum(1 for v in answers.values() if str(v).strip())
    print(f"  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if GITHUB_USERNAME.strip():
        print(f"  Student  : @{GITHUB_USERNAME.strip()}")
    print(f"  Grading  : please wait...\n")

    # Build the prompt — questions + student answers
    lines = []
    for i, (k, v) in enumerate(answers.items(), 1):
        a = str(v).strip() or "(no answer)"
        lines.append(f"Q{i}: {a}")
    qa_block = "\n".join(lines)

    prompt = f"""You are grading a student quiz for the "{_NB_TITLE}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP).

The student answered {nd}/{len(answers)} questions:

{qa_block}

For each question, tell the student:
- Whether they are CORRECT, PARTIALLY CORRECT, or INCORRECT
- One specific sentence of feedback (what they got right, or exactly what concept to review)

Accept any reasonable paraphrase as CORRECT. Do not require exact wording.

End with:
- An overall grade: Excellent / Good / Needs Review / Incomplete  
- A score out of {len(answers)}
- One sentence on what to review if they struggled, or encouragement if they did well

Be direct, specific, and encouraging."""

    # Call Gemini via google.generativeai — uses Colab's built-in auth
    result_text = None
    try:
        import google.generativeai as genai
        from google.colab import auth
        auth.authenticate_user()
        genai.configure()
        model  = genai.GenerativeModel("gemini-2.0-flash")
        resp   = model.generate_content(prompt)
        result_text = resp.text
    except Exception as e1:
        # Fallback: try with userdata key if student stored one
        try:
            from google.colab import userdata, auth
            key = userdata.get("GEMINI_API_KEY")
            import google.generativeai as genai
            genai.configure(api_key=key)
            resp = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
            result_text = resp.text
        except Exception as e2:
            result_text = None

    if result_text:
        print("\u2550"*56)
        print(f"  \U0001f916 AI Feedback \u2014 {_NB_TITLE}")
        print("\u2550"*56)
        if GITHUB_USERNAME.strip():
            print(f"  Student: @{GITHUB_USERNAME.strip()}\n")
        # Print the response cleanly, wrapped
        for para in result_text.strip().split("\n"):
            para = para.strip()
            if not para: print(); continue
            for line in textwrap.wrap(para, 56):
                print(f"  {line}")
        print("\n" + "\u2550"*56)
    else:
        print("  Gemini is temporarily unavailable.")
        print("  Your answers are saved — re-run this cell in a minute to get feedback.")

    print("\n  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


# ⚖️ Bias-Variance Tradeoff
**ISLP Chapter 2 · Pattern Recognition for the Rest of Us**

> The single most important concept in machine learning. Every modeling decision you make — model complexity, regularization, ensemble size — is a bias-variance tradeoff in disguise.

---
### What you'll learn
- What bias and variance mean intuitively and mathematically
- Why the total test error = Bias² + Variance + Irreducible error
- How to *see* the tradeoff in learning curves
- How this explains overfitting and underfitting

### Time
~50 minutes

## 🎯 Part 1 — The Core Idea

Imagine you're throwing darts at a target. You throw 10 sets of darts, each set trained on a different sample of the same data.

- **High Bias** = all darts clustered together, but far from the bullseye. The model systematically misses — it's too simple to capture the true pattern. This is **underfitting**.
- **High Variance** = darts spread all over the target. The model is very sensitive to which training data it saw. Highly flexible models tend this way. This is **overfitting**.
- **Low Bias + Low Variance** = darts clustered on the bullseye. This is what we want.

The tradeoff: as you increase model flexibility, bias decreases but variance increases. There's a sweet spot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,
    'axes.spines.top':False,'axes.spines.right':False,'font.size':11
})

# ── Dart board illustration ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
scenarios = [
    ('High Bias\nLow Variance', [(0.3,0.3),(0.35,0.28),(0.32,0.33),(0.28,0.31),(0.33,0.35),(0.29,0.29)], '#e85d20'),
    ('Low Bias\nHigh Variance', [(-0.1,0.4),(0.5,-0.3),(-0.4,-0.2),(0.3,0.5),(-0.2,-0.4),(0.4,0.1)], '#c0392b'),
    ('High Bias\nHigh Variance', [(0.4,0.3),(0.1,-0.4),(0.5,0.1),(-0.3,0.4),(0.3,-0.2),(0.4,0.5)], '#888'),
    ('Low Bias\nLow Variance ✓', [(0.05,0.03),(-0.04,0.06),(0.02,-0.05),(0.06,0.04),(-0.03,-0.02),(0.04,0.05)], '#1a7a45'),
]

for ax, (title, points, color) in zip(axes, scenarios):
    for r, alpha in [(1,'#f0f0f0'),(0.6,'#e0e8f0'),(0.3,'#c8d8f0'),(0.1,'#a0c0e8')]:
        circle = plt.Circle((0,0), r, color=alpha, zorder=1)
        ax.add_patch(circle)
    ax.plot(0, 0, 'k+', markersize=12, zorder=3, lw=2)
    xs, ys = zip(*points)
    ax.scatter(xs, ys, color=color, s=60, zorder=4, edgecolors='white', lw=0.5)
    ax.set_xlim(-1.2,1.2); ax.set_ylim(-1.2,1.2)
    ax.set_aspect('equal'); ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Bias-Variance as a Dart Board', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()
print("📌 We want bottom-right: low bias (near bullseye) + low variance (tight cluster).")

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Advertising

In [ ]:
import pandas as pd
ads = pd.read_csv(f'{DATA_DIR}/Advertising.csv')
print(f"Advertising: {ads.shape}  |  Columns: {list(ads.columns)}")
ads.head()

## 🧮 Part 2 — The Math

For any point x₀, the expected test MSE decomposes as:

```
E[(y₀ - f̂(x₀))²] = Var(f̂(x₀)) + [Bias(f̂(x₀))]² + Var(ε)
                  =   Variance   +      Bias²        + Irreducible
```

Where:
- **Variance** = how much f̂ changes across different training sets. High-flexibility models (deep trees, high-degree polynomials) have high variance.
- **Bias** = the error from wrong assumptions. A linear model fit to nonlinear data has high bias — it's systematically wrong.
- **Var(ε)** = irreducible noise — fixed by the data, not the model.

The tradeoff: decreasing bias (by increasing flexibility) tends to increase variance, and vice versa. Total error is U-shaped as a function of complexity.

In [ ]:
# ── Simulate the bias-variance tradeoff exactly ─────────────────────────────
np.random.seed(42)

# True function: a smooth nonlinear curve
def f_true(x): return np.sin(2*np.pi*x)

n_train = 30
n_reps  = 200   # number of different training sets
sigma   = 0.3   # noise level (irreducible error std)
x_test  = np.linspace(0, 1, 100)

degrees = [1, 2, 3, 5, 8, 12]
bias2_list, var_list, total_list = [], [], []

for d in degrees:
    preds = []
    for _ in range(n_reps):
        x_tr = np.random.uniform(0, 1, n_train)
        y_tr = f_true(x_tr) + np.random.normal(0, sigma, n_train)
        pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
        pipe.fit(x_tr.reshape(-1,1), y_tr)
        preds.append(pipe.predict(x_test.reshape(-1,1)))
    
    preds = np.array(preds)             # shape: (n_reps, 100)
    mean_pred = preds.mean(axis=0)      # average prediction at each x
    
    bias2   = np.mean((mean_pred - f_true(x_test))**2)
    variance = np.mean(preds.var(axis=0))
    irred   = sigma**2
    
    bias2_list.append(bias2)
    var_list.append(variance)
    total_list.append(bias2 + variance + irred)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: bias² and variance separately
axes[0].plot(degrees, bias2_list,  'o-', color='#e85d20', lw=2, label='Bias²', markersize=6)
axes[0].plot(degrees, var_list,    's-', color='#1e5fa8', lw=2, label='Variance', markersize=6)
axes[0].axhline(sigma**2, color='#888', lw=1.5, ls=':', label=f'Irreducible ({sigma**2:.2f})')
axes[0].set_xlabel('Model Complexity (degree)')
axes[0].set_ylabel('Error Component')
axes[0].set_title('Bias² and Variance vs Complexity')
axes[0].legend()

# Right: total expected test error
best_d = degrees[np.argmin(total_list)]
axes[1].plot(degrees, total_list, 'o-', color='#1a7a45', lw=2.5, label='Total Test Error', markersize=6)
axes[1].axvline(best_d, color='#e85d20', lw=1.5, ls='--', label=f'Sweet spot (degree {best_d})')
axes[1].set_xlabel('Model Complexity (degree)')
axes[1].set_ylabel('Expected Test MSE')
axes[1].set_title('U-shaped Test Error Curve')
axes[1].legend()

plt.suptitle('The Bias-Variance Tradeoff', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n📌 Sweet spot at degree {best_d}.")
print(f"   Degree 1: Bias²={bias2_list[0]:.3f}, Var={var_list[0]:.3f}  → underfitting (high bias)")
print(f"   Degree {degrees[-1]}: Bias²={bias2_list[-1]:.3f}, Var={var_list[-1]:.3f}  → overfitting (high variance)")

## 👁️ Part 3 — Seeing It in Action: Learning Curves

A **learning curve** plots training and validation error as training set size increases.

- High-bias model: both curves plateau at a high error — more data won't help much
- High-variance model: training error is low, validation error is high — gap closes as n grows

Learning curves tell you *what's wrong* with your model and what to do about it.

In [ ]:
# ── Learning curves for underfit, good fit, overfit ─────────────────────────
np.random.seed(1)
n_total = 200
X_full = np.random.uniform(0, 1, n_total)
Y_full = f_true(X_full) + np.random.normal(0, sigma, n_total)

train_sizes = range(10, 160, 10)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
model_specs = [(1, 'Degree 1 — Underfit (High Bias)'),
               (3, 'Degree 3 — Good Fit'),
               (12, 'Degree 12 — Overfit (High Variance)')]
model_colors = ['#e85d20', '#1a7a45', '#c0392b']

for ax, (deg, title), color in zip(axes, model_specs, model_colors):
    tr_errs, val_errs = [], []
    for n in train_sizes:
        X_tr, Y_tr = X_full[:n], Y_full[:n]
        X_val, Y_val = X_full[160:], Y_full[160:]
        pipe = Pipeline([('poly',PolynomialFeatures(deg)),('lr',LinearRegression())])
        pipe.fit(X_tr.reshape(-1,1), Y_tr)
        tr_errs.append(mean_squared_error(Y_tr, pipe.predict(X_tr.reshape(-1,1))))
        val_errs.append(mean_squared_error(Y_val, pipe.predict(X_val.reshape(-1,1))))
    
    ax.plot(list(train_sizes), tr_errs,  '-', color=color,    lw=2, label='Training error')
    ax.plot(list(train_sizes), val_errs, '--', color='#1e5fa8', lw=2, label='Validation error')
    ax.axhline(sigma**2, color='#aaa', lw=1, ls=':', label='Irreducible floor')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Training set size')
    if ax == axes[0]: ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 0.35)

plt.suptitle('Learning Curves — Diagnosing Bias vs Variance', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("\n📌 High bias (left): both curves meet at high error — model can't learn the pattern.")
print("   Good fit (middle): curves converge near irreducible floor.")
print("   High variance (right): big gap between training and validation — needs more data or regularization.")

## 💼 Exercise

**Task 1:** Using the simulation code above, change `sigma` from 0.3 to 0.8. How does the bias-variance decomposition change? Why?

**Task 2:** Load the Advertising dataset and fit polynomial models of degrees 1, 2, 4, 8 predicting Sales from TV. Use an 80/20 split. Plot training vs test MSE. At which degree does test MSE start to rise?

**Task 3:** A colleague says: *'My model has 98% accuracy on training data, so it must be good.'* Using the concepts from this notebook, explain in 3 sentences why they might be wrong.

In [ ]:
# @title 📝 Quiz — Bias-Variance Tradeoff
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** As model complexity increases, what happens to bias?
# @markdown **Q2:** As model complexity increases, what happens to variance?
# @markdown **Q3:** What does the irreducible error floor represent?
# @markdown **Q4:** A model with high training accuracy but poor test accuracy suffers from what?
# @markdown **Q5:** Name one practical way to reduce variance without changing the model family

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results